# Run subspace ablations across all new-activation models

Runs `run_ablation.py` (subspace mode, rank=1) for the four models that have both new activations and SVD subspaces:

| Tag | Model | Activations | SVD dir |
| --- | --- | --- | --- |
| `gemma3_4b`  | `google/gemma-3-4b-it`             | `code/activations/gemma3acts/4b`  | `code/svd/svd/gemma-3-4b-it` |
| `gemma4_E2B` | `google/gemma-3n-E2B-it`           | `code/activations/gemma4acts/E2B` | `code/svd/gemma-4-E2B-it`    |
| `llama3_3b`  | `meta-llama/Llama-3.2-3B-Instruct` | `code/activations/llama3acts`     | `code/svd/llama-3.2-3b`      |
| `qwen35_4b`  | `Qwen/Qwen3-4B`                    | `code/activations/qwen35acts`     | `code/svd/qwen-3.5-4b`       |

Results are written to Google Drive at `MyDrive/safety-utility-results/<tag>_subspace_r1/`. The HF weights cache is wiped after each run so disk doesn't fill up.

**Before running:** set the runtime to GPU (Runtime → Change runtime type → GPU). T4 should be enough for these 3B–4B models in bf16/fp16.

## 1. Install git-lfs and check GPU

In [ ]:
!nvidia-smi | head -n 20
!apt-get -qq install -y git-lfs
!git lfs install

## 2. Clone the repo and pull LFS activations

**Replace `REPO_URL` with your fork's URL.** After `git lfs pull` the activation `.npy` files should be hundreds of MB; if they're 134 bytes the LFS pull didn't work.

In [ ]:
REPO_URL = "https://github.com/<your-user>/safety-utility-disentanglement.git"
!git clone $REPO_URL
%cd safety-utility-disentanglement
!git lfs pull
!ls -la code/activations/llama3acts code/activations/qwen35acts code/activations/gemma3acts/4b code/activations/gemma4acts/E2B

## 3. Install Python deps

In [ ]:
!pip -q install -U transformers accelerate datasets scikit-learn

## 4. Hugging Face login

Gemma and Llama are gated — use a token that has access to `google/gemma-*` and `meta-llama/*`.

In [ ]:
from huggingface_hub import login
login()

## 5. Mount Drive

Results are saved here so they survive a Colab disconnect.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RESULTS_ROOT = "/content/drive/MyDrive/safety-utility-results"
!mkdir -p $RESULTS_ROOT
print("results going to:", RESULTS_ROOT)

## 6. Run all four ablations

After each run we delete that model's weights from `~/.cache/huggingface` to free disk before the next download.

In [ ]:
import os, shutil, subprocess, gc
from pathlib import Path

os.chdir("/content/safety-utility-disentanglement/code/ablation")

RESULTS_ROOT = "/content/drive/MyDrive/safety-utility-results"

runs = [
    # (model_id, act_dir, suffix, svd_dir, svd_tag, out_tag)
    ("google/gemma-3-4b-it",             "../activations/gemma3acts/4b",  "4b_n500",
     "../svd/svd/gemma-3-4b-it",         "gemma-3-4b-it",  "gemma3_4b"),
    ("google/gemma-3n-E2B-it",           "../activations/gemma4acts/E2B", "E2B",
     "../svd/gemma-4-E2B-it",            "gemma-4-E2B-it", "gemma4_E2B"),
    ("meta-llama/Llama-3.2-3B-Instruct", "../activations/llama3acts",     "3b",
     "../svd/llama-3.2-3b",              "llama-3.2-3b",   "llama3_3b"),
    ("Qwen/Qwen3-4B",                    "../activations/qwen35acts",     "4b",
     "../svd/qwen-3.5-4b",               "qwen-3.5-4b",    "qwen35_4b"),
]

for model, act_dir, suffix, svd_dir, svd_tag, out_tag in runs:
    out_dir = f"{RESULTS_ROOT}/{out_tag}_subspace_r1"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "run_ablation.py",
        "--model", model,
        "--activations-dir", act_dir,
        "--suffix", suffix,
        "--direction-source", "subspace",
        "--svd-dir", svd_dir,
        "--svd-tag", svd_tag,
        "--rank", "1",
        "--n", "50",
        "--out-dir", out_dir,
    ]
    print("\n" + "=" * 80)
    print(">>>", " ".join(cmd))
    print("=" * 80, flush=True)
    rc = subprocess.run(cmd).returncode
    print(f"[{out_tag}] return code = {rc}")

    # free disk: drop this model's HF weights cache
    org, name = model.split("/")
    cache_dir = Path.home() / ".cache" / "huggingface" / "hub" / f"models--{org}--{name}"
    if cache_dir.exists():
        print(f"removing {cache_dir}")
        shutil.rmtree(cache_dir, ignore_errors=True)

    # free VRAM in case anything lingers in this Python process
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    !df -h /root | tail -n 1

## 7. Aggregate all summaries into one CSV

In [ ]:
import json, glob
import pandas as pd

RESULTS_ROOT = "/content/drive/MyDrive/safety-utility-results"
rows = []
for path in sorted(glob.glob(f"{RESULTS_ROOT}/*/summary.json")):
    with open(path) as f:
        s = json.load(f)
    row = {
        "tag": path.split("/")[-2],
        "model": s["model"],
        "safety_layer": s["best_safety_layer"],
        "utility_layer": s["best_utility_layer"],
        "safety_AUROC": round(s["safety_auroc_at_layer"], 3),
        "utility_AUROC": round(s["utility_auroc_at_layer"], 3),
    }
    row.update({k: round(v, 3) for k, v in s["refusal_rates"].items()})
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = f"{RESULTS_ROOT}/all_summaries.csv"
df.to_csv(csv_path, index=False)
print("wrote", csv_path)
df